In [18]:
# Run python script
import subprocess
import sys

args = ['hello']
kwargs = {'--n':'5'}
subprocess.run(
    [sys.executable, './delayed_message.py', 'Hello', '--n', '2'],
    check=True,
)

Hello


CompletedProcess(args=['/home/bashar/Study/Projects/AIVoiceAssistant/venv/bin/python', './delayed_message.py', 'Hello', '--n', '2'], returncode=0)

In [19]:
from langchain_ollama import ChatOllama
from langchain.agents import create_agent
from langchain_core.messages import SystemMessage, HumanMessage, ToolMessage, AIMessage
from langchain_core.tools import tool, Tool
from pydantic import BaseModel


import subprocess
import sys 

In [20]:

class ExecutePythonScriptInput(BaseModel):
    path: str
    args: list[str] | None = None
    kwargs: dict[str, str | int | float] | None = None


@tool(args_schema=ExecutePythonScriptInput)
def execute_python_script(
    path: str,
    args: list[str] | None = None,
    kwargs: dict[str, str | int | float] | None = None,
) -> str:
    """Run a Python script with positional args and named CLI options.

    Args:
        path: Path to the Python script.
        args: Positional command-line arguments.
        kwargs: Options with values, e.g. {"n": 3} becomes --n 3.
            Keys may include their leading dashes, e.g. --n or -n.
    """
    command = [sys.executable, path, *[str(arg) for arg in (args or [])]]
    for name, value in (kwargs or {}).items():
        option = name if name.startswith("-") else f"--{name}"
        command.extend([option, str(value)])
    result = subprocess.run(
        command,
        check=True,
        capture_output=True,
        text=True,
    )
    return result.stdout


@tool
def write_python_script(path: str , content: str):
    "a function to write a python script by specifyign the path and the content"
    with open(path, 'w') as f:
        f.write(content)

In [21]:
chat = ChatOllama(
    model='qwen3.5:4b',
    temperature=0,
    max_tokens=256,
)

SystemPrompt = '''
You are a helpful assistant, the user will ask you to perform some task, you should find the appropriate tool to run the task.
### TOOLS:
- list_python_scripts: is used to list available scripts
- write_python_script: is used to write a specific python script to a specific directory.
- execute_python_script: is used to run a pre-defined python script
'''
tools = [execute_python_script, write_python_script]

messages = [HumanMessage(content='run a python script to print the word Hell after 3 seconds')]
messages_write_python_script = [HumanMessage(content='write a python script that takes two args and returns the sum, put the file in agent_generated_scripts directory and namd it proprely')]

def llm_chat(messages:list, tools: list | None = None) -> list: 
    tools_dict = {tool.name: tool for tool in tools}
    result = chat.bind_tools(tools).invoke([SystemMessage(content=SystemPrompt)] + messages)
    messages.append(result)
    if result.content:
        print(f"[AGENT]: {result.content}")
    if result.content == '' and result.tool_calls:
        for call in result.tool_calls:
            print(f"[TOOL CALL]: {call}")
            tool_call = tools_dict[call['name']]
            tool_result = tool_call.invoke(call)
            print(f"[OBS]: {tool_result.content}")
            messages.append(tool_result)
    return messages

In [22]:
result = llm_chat(messages_write_python_script, tools)

HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


[TOOL CALL]: {'name': 'write_python_script', 'args': {'path': 'agent_generated_scripts/sum_calculator.py', 'content': '#!/usr/bin/env python3\n"""Sum calculator that takes two arguments and returns their sum."""\n\ndef calculate_sum(a, b):\n    """Calculate the sum of two numbers."""\n    return a + b\n\nif __name__ == "__main__":\n    import sys\n    \n    if len(sys.argv) != 3:\n        print("Usage: python sum_calculator.py <num1> <num2>")\n        sys.exit(1)\n    \n    try:\n        num1 = float(sys.argv[1])\n        num2 = float(sys.argv[2])\n        result = calculate_sum(num1, num2)\n        print(f"Sum of {num1} and {num2} is: {result}")\n    except ValueError:\n        print("Error: Please provide two valid numbers as arguments.")\n        sys.exit(1)'}, 'id': '5becf128-0f53-49cb-a0bd-36ab7cb3edbd', 'type': 'tool_call'}
[OBS]: null


#### OKAY, we have a primitive (but working) llm agent, now let's add the ASR model:

In [23]:
from qwen_asr import Qwen3ASRModel
import torch

model = Qwen3ASRModel.from_pretrained(
    "Qwen/Qwen3-ASR-0.6B",
    dtype=torch.bfloat16,
    device_map="cuda:0",
    # attn_implementation="flash_attention_2",
    max_inference_batch_size=32, # Batch size limit for inference. -1 means unlimited. Smaller values can help avoid OOM.
    max_new_tokens=256, # Maximum number of tokens to generate. Set a larger value for long audio input.
)



In [24]:
## Record 5-second audio
import sounddevice as sd
from scipy.io.wavfile import write

sample_rate = 24000
duration = 5

def record_audio(sr, duration) :
    print(f"Recroding audio for {duration} seconds...")
    audio = (sd.rec(
        int(duration * sample_rate),
        samplerate=sample_rate,
        channels=1,
        dtype="int16",
    ), sample_rate)
    sd.wait()
    return audio

In [25]:
%time
results = model.transcribe(
    audio=audio,
    language="English"
)
print(results)

Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


CPU times: user 0 ns, sys: 14 μs, total: 14 μs
Wall time: 25 μs
[ASRTranscription(language='English', text='Hello, what is the capital of France?', time_stamps=None)]


### Combine audio -> ASR -> LLM -> Output

In [ ]:
audio = record_audio(sr=24000, duration=10)
text = model.transcribe(audio=audio, language="English")[0].text
messages = [HumanMessage(content= text)]
llm_result = llm_chat(messages=messages, tools=tools)


Recroding audio for 5 seconds...


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


[AGENT]: The capital of France is Paris. It's located in north-central France and has been the country's capital since 1338. Paris is known for its iconic landmarks like the Eiffel Tower, Louvre Museum, Notre-Dame Cathedral, and its famous fashion, culinary, and art scenes.


In [27]:
llm_result

[HumanMessage(content='Hello there. Can you tell me what is the capital of France?', additional_kwargs={}, response_metadata={}),
 AIMessage(content="The capital of France is Paris. It's located in north-central France and has been the country's capital since 1338. Paris is known for its iconic landmarks like the Eiffel Tower, Louvre Museum, Notre-Dame Cathedral, and its famous fashion, culinary, and art scenes.", additional_kwargs={}, response_metadata={'model': 'qwen3.5:4b', 'created_at': '2026-10-17T10:14:01.152303828Z', 'done': True, 'done_reason': 'stop', 'total_duration': 1612378567, 'load_duration': 738586, 'prompt_eval_count': 520, 'prompt_eval_duration': 210276000, 'eval_count': 119, 'eval_duration': 1397834000, 'logprobs': None, 'model_name': 'qwen3.5:4b', 'model_provider': 'ollama'}, id='lc_run--01a1495a-a472-7da0-b9e0-4e0c1607b14f-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 520, 'output_tokens': 119, 'total_tokens': 639})]